# DuckDB + Horizon Iceberg REST Catalog

Query Snowflake-managed silver Dynamic Iceberg Tables from DuckDB using Snowflake's
Horizon Iceberg REST Catalog (HIRC) and a Programmatic Access Token (PAT).

**Before running:** complete the Dynamic Iceberg Tables chapter so all five `dt_*` tables
exist in `balloon_silver.silver` and have refreshed at least once.

In [ ]:
# Configuration defaults — all values are read from .env first;
# these serve as fallbacks when the env vars are unset.
#
# Set in .env before running:
#   SNOWFLAKE_ACCOUNT_URL=https://<org>-<account>.snowflakecomputing.com
#   SA_ROLE=duckdb_silver_reader           # no hyphens (HIRC requirement)
#   SNOWFLAKE_SILVER_DATABASE=balloon_silver
#   SNOWFLAKE_PASSWORD=<paste-pat-token>   # PAT value shown once at creation

DEFAULT_SA_ROLE = 'duckdb_silver_reader'
DEFAULT_DATABASE = 'balloon_silver'

## Prerequisites

1. Silver DTs exist in `balloon_silver.SILVER` and have refreshed at least once.
2. Snowflake service account role `duckdb_silver_reader` has been created with SELECT
   on all silver DTs (see sfguide DuckDB Integration chapter).
3. Generate a PAT for `duckdb_sa` and store it in your OS keychain:
   ```bash
   task snowflake:pat-create
   ```
   Then print the token value to your console:
   ```bash
   task snowflake:pat-print
   ```
4. `.env` (repo root) contains:
   ```
   SNOWFLAKE_ACCOUNT_URL=https://<org>-<account>.snowflakecomputing.com
   SA_USER=duckdb_sa
   SA_ROLE=duckdb_silver_reader
   SNOWFLAKE_PASSWORD=<paste output of: task snowflake:pat-print>
   SNOWFLAKE_SILVER_DATABASE=balloon_silver
   ```
5. Never commit `.env` — it is listed in `.gitignore`.

> **Role name rule:** HIRC does not support hyphens in role names.
> Use `duckdb_silver_reader` (underscores), not `duckdb-silver-reader`.

In [ ]:
import duckdb
from dotenv import find_dotenv, load_dotenv
import os
import traceback

load_dotenv(find_dotenv())

pat_token = os.getenv('SNOWFLAKE_PASSWORD')
snowflake_account_url = os.getenv('SNOWFLAKE_ACCOUNT_URL', '').rstrip('/')
sa_role = os.getenv('SA_ROLE', DEFAULT_SA_ROLE)
database = os.getenv('SNOWFLAKE_SILVER_DATABASE', DEFAULT_DATABASE)

if not pat_token:
    raise ValueError('SNOWFLAKE_PASSWORD (PAT token) is not set — add it to .env')
if not snowflake_account_url:
    raise ValueError('SNOWFLAKE_ACCOUNT_URL is not set — add it to .env')

catalog_uri = snowflake_account_url.lower() + '/polaris/api/catalog'

print(f'Catalog URI : {catalog_uri}')
print(f'Database    : {database}')
print(f'Role        : {sa_role}')

In [ ]:
# Install and load the DuckDB Iceberg and HTTPFS extensions.
# Extensions are cached after first install — subsequent runs are fast.
conn = duckdb.connect()
conn.execute('INSTALL iceberg;')
conn.execute('LOAD iceberg;')
conn.execute('INSTALL httpfs;')
conn.execute('LOAD httpfs;')
print('Extensions loaded.')

## Connect to Horizon Iceberg REST Catalog

DuckDB uses a PAT-based Iceberg secret to authenticate via the OAuth2 client credentials
flow, then attaches the `balloon_silver` database. Snowflake vends temporary cloud
credentials so DuckDB can read S3 data files directly — no data proxying.

In [ ]:
secret_sql = f"""
  CREATE OR REPLACE SECRET iceberg_pat_secret (
    TYPE iceberg,
    CLIENT_ID '',
    CLIENT_SECRET '{pat_token}',
    OAUTH2_SERVER_URI '{catalog_uri}/v1/oauth/tokens',
    OAUTH2_GRANT_TYPE 'client_credentials',
    OAUTH2_SCOPE 'session:role:{sa_role}'
  );
"""

attach_sql = f"""
  ATTACH '{database}' AS {database} (
    TYPE iceberg,
    SECRET iceberg_pat_secret,
    ENDPOINT '{catalog_uri}',
    SUPPORT_NESTED_NAMESPACES false
  );
"""

try:
    conn.execute(secret_sql)
    conn.execute(attach_sql)
    print(f'Attached {database} via HIRC.')
except Exception:
    traceback.print_exc()

## Discover Tables

Snowflake identifiers are **UPPERCASE** when accessed through HIRC.
Use `SILVER.DT_PLAYER_LEADERBOARD` (uppercase), not `silver.dt_player_leaderboard`.

In [ ]:
tables = conn.execute('SHOW ALL TABLES').fetchall()
print(f'Found {len(tables)} table(s):')
for t in tables:
    db, schema, name = t[0], t[1], t[2]
    print(f'  {db}.{schema}.{name}')

## Query Silver Dynamic Iceberg Tables

Each cell queries one of the five silver DTs. All identifiers use uppercase schema
and table names as required by HIRC.

In [ ]:
# dt_player_leaderboard — per-player total score and bonus pops
try:
    df = conn.execute(f"""
        SELECT player, total_score, bonus_pops, last_event_ts
        FROM {database}.SILVER.DT_PLAYER_LEADERBOARD
        ORDER BY total_score DESC NULLS LAST
        LIMIT 10
    """).df()
    print('dt_player_leaderboard:')
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_balloon_color_stats — per-player, per-color breakdown
try:
    df = conn.execute(f"""
        SELECT player, balloon_color, balloon_pops, points_by_color, bonus_hits
        FROM {database}.SILVER.DT_BALLOON_COLOR_STATS
        ORDER BY player, points_by_color DESC NULLS LAST
        LIMIT 10
    """).df()
    print('dt_balloon_color_stats:')
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_realtime_scores — 15-second windowed scores per player
try:
    df = conn.execute(f"""
        SELECT player, total_score, window_start, window_end
        FROM {database}.SILVER.DT_REALTIME_SCORES
        ORDER BY window_start DESC, player
        LIMIT 10
    """).df()
    print('dt_realtime_scores:')
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_balloon_colored_pops — 15-second windows by player and balloon color
try:
    df = conn.execute(f"""
        SELECT player, balloon_color, balloon_pops, window_start, window_end
        FROM {database}.SILVER.DT_BALLOON_COLORED_POPS
        ORDER BY window_start DESC, player, balloon_color
        LIMIT 10
    """).df()
    print('dt_balloon_colored_pops:')
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_color_performance_trends — avg score per pop by color over 15-second windows
try:
    df = conn.execute(f"""
        SELECT balloon_color, avg_score_per_pop, total_pops, window_start, window_end
        FROM {database}.SILVER.DT_COLOR_PERFORMANCE_TRENDS
        ORDER BY window_start DESC, avg_score_per_pop DESC NULLS LAST
        LIMIT 10
    """).df()
    print('dt_color_performance_trends:')
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

## Cleanup

Detach the database when done to close the HIRC connection.

In [ ]:
try:
    conn.execute(f'DETACH {database}')
    print(f'Detached {database}.')
except Exception as e:
    print(f'Detach skipped (may already be detached): {e}')